# Ride classifier: training and inference benchmark on a Colab GPU

AI Infrastructure, step 4. Same code and protocol as on the laptop (`ride_classifier.train_benchmark`, `ride_classifier.benchmark`): warm-up not timed, CUDA synchronised, 3 repeats, raw CSV rows. **Measure, do not claim**: if a cell fails (no GPU, install error), keep the error and report it.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. On the laptop: `make colab-bundle` creates `colab_bundle.zip` (train and test splits + model v1, ~27 MB). Keep it private: it contains course data.

Results are downloaded at the end as `colab_results.zip`; unzip it into `ai-infrastructure/benchmarks/` on the laptop.

In [ ]:
!nvidia-smi
import platform, torch
print('Python', platform.python_version(), '| PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
# Code: the public repository. Colab runs Python 3.12 and already has a CUDA build of PyTorch,
# so the project is NOT pip-installed (it requires 3.14): its src/ folder is used directly.
!git clone -q https://github.com/ngilhodes-netizen/MLOps-AI-infra-Project.git repo
%cd repo
!pip -q install mlflow==3.18.0 pyarrow
import os, sys
sys.path.insert(0, 'src')
os.environ['PYTHONPATH'] = 'src'
os.environ['MLFLOW_DISABLE_AGENT_HINT'] = '1'

In [ ]:
# Data + model: upload colab_bundle.zip (made on the laptop with `make colab-bundle`)
from google.colab import files
files.upload()
!unzip -o -q colab_bundle.zip && ls -la data/processed model

In [ ]:
# Environment record of THIS machine (GPU, CPU, versions, model and test hashes)
!python -m ride_classifier.environment --out colab_results
!cat colab_results/environment.md

In [ ]:
# Training on the Colab CPU, then on the GPU: same epochs, same batch size, 3 repeats each
!python -m ride_classifier.train_benchmark --device cpu  --experiment training-colab --out colab_results/raw/training.csv
!python -m ride_classifier.train_benchmark --device cuda --experiment training-colab --out colab_results/raw/training.csv

In [ ]:
# Inference without HTTP on the GPU (the network on CUDA; preprocessing stays on CPU, as it would in production)
!python -m ride_classifier.benchmark run --device cuda --experiment inference-colab --out colab_results/raw/inference-colab.csv
!python -m ride_classifier.benchmark run --device cpu  --experiment inference-colab --out colab_results/raw/inference-colab.csv

In [ ]:
# Medians, then download everything (raw rows + environment)
!python -m ride_classifier.benchmark summarize --raw colab_results/raw/training.csv --out colab_results/training-colab-results.csv
!python -m ride_classifier.benchmark summarize --raw colab_results/raw/inference-colab.csv --out colab_results/inference-colab-results.csv
!zip -q -r colab_results.zip colab_results
files.download('colab_results.zip')